# TAG-MTL-DR v2 — complete pipeline for Google Colab

CNN lesion tokens + lesion-aware vessel graph (GAT) + optic-disc-rooted vessel-path GRU +
cross-attention + CORAL + auxiliary vessel segmentation, for diabetic retinopathy grading, with a
single-protocol benchmark (reproduced baselines + ablations), external validation,
statistics, LaTeX tables and figures for a journal article.

**How to use**
1. `Runtime > Change runtime type > GPU` (T4 or better).
2. Run the cells **in order**.
3. Set `DEMO = True` first: the notebook builds a tiny synthetic dataset and checks the
   whole chain in a few minutes. Its numbers are meaningless.
4. Then set `DEMO = False`, point `DATA_DIR` to your images on Google Drive, build the
   manifest (cell 5) and run the benchmark.

No result is included: every number comes from your own runs.

## 1. Installation

In [ ]:
!pip -q install timm torch-geometric scikit-image scikit-learn scipy matplotlib pillow
import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (enable a GPU runtime)")

## 2. Settings

In [ ]:
DEMO = True                     # True: synthetic check; False: your data
USE_DRIVE = not DEMO            # mount Google Drive to read images and save results

# --- paths (used when DEMO = False) ---
WORK_DIR = "/content/drive/MyDrive/TAG_MTL_DR"        # results are saved here
DATA_DIR = "/content/drive/MyDrive/datasets"          # where your images are

# --- experiment size ---
QUICK = True                    # QUICK: fewer methods/seeds for a free Colab GPU; False: full paper protocol
if QUICK:
    METHODS = ["EfficientNet-B3", "EfficientNet-B3 + CORAL", "ConvNeXt-T", "TAG-MTL-DR (ours)",
               "w/o vessel-path GRU", "w/o lesion-aware nodes"]
    SEEDS, FOLDS, EPOCHS = [42], 5, 15
else:
    METHODS = ["all"]           # 7 baselines + ours + 6 ablations
    SEEDS, FOLDS, EPOCHS = [42, 43, 44], 5, 30
BATCH, WORKERS = 16, 2
CACHE_DIR = "/content/cache"    # pre-processing cache on the fast local disk (rebuilt automatically in a new session)
SAVE_MODELS = True              # keep checkpoints (needed by explain.py)

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    WORK_DIR = "/content/tag_mtl_dr_demo"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs("/content/code", exist_ok=True)
%cd /content/code
print("results will be written to", WORK_DIR)

## 3. Source code (model, benchmark, analysis)
These cells write the three modules to `/content/code`.

In [ ]:
%%writefile tag_mtl_dr.py
"""TAG-MTL-DR v2: lesion-aware vessel graph + vessel-path recurrence for DR grading.

Components (each can be switched off for the ablation study):

1. CNN lesion branch: EfficientNet-B3 feature map, one token per cell.
2. Lesion-aware vessel graph (GNN/GAT): nodes are the end-points and junctions of the
   vessel skeleton, edges the vessel segments (length, tortuosity). Each node carries its
   geometry *and* the CNN feature sampled at its position, so the graph attention network
   propagates lesion evidence along the vessels.
3. Vessel-path recurrence (RNN): the vessel tree is rooted at the optic disc; every
   root-to-leaf path is read in anatomical order by a bidirectional GRU over the GAT node
   embeddings, and the paths are pooled by attention. It models how lesions and geometry
   change along a vessel, from the disc to the periphery.
4. Cross-attention from lesion tokens to graph nodes and path embeddings (residual).
5. CORAL ordinal head (rank-consistent) and an auxiliary U-Net vessel segmentation (multi-task).

Speed: images, vessel masks, optic-disc positions, graphs and paths are computed once
(``build_cache``, multi-process) and read from a memory-mapped cache; resizing, flips and
normalisation run on the GPU. Vessel graphs come from a deterministic vessel extraction
applied to every image (training and test alike), so no CPU work is left in the training loop.

Novelty note: to our knowledge, sampling CNN lesion features onto a vessel-topology graph
and reading optic-disc-to-periphery vessel paths with a recurrent network have not been
combined for DR grading; verify with a literature search before claiming it.

``python tag_mtl_dr.py --smoke`` checks every component on random data.
"""

from __future__ import annotations

import argparse
import json
import os
from collections import deque
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from skimage import exposure, morphology
from skimage.filters import threshold_otsu
from skimage.morphology import skeletonize
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.utils import softmax as seg_softmax
from torch_geometric.utils import to_dense_batch

NUM_CLASSES = 5
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
NODE_FEATS = 7          # y, x, degree/4, is_end, is_junction, distance to optic disc, is_root
EDGE_FEATS = 2          # segment length, tortuosity
_OFFS = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


# ============================================================ classical vessel extraction
def pseudo_vessel_mask(rgb: np.ndarray, radius: int = 7, min_size: int = 50) -> np.ndarray:
    """Classical vessel mask (green channel, CLAHE, black top-hat, Otsu)."""
    g = rgb[..., 1].astype(np.float32)
    g = g / 255.0 if g.max() > 1.0 else g
    g = exposure.equalize_adapthist(g, clip_limit=0.02)
    th = morphology.black_tophat(g, morphology.disk(radius))   # vessels are darker than background
    if th.max() <= 0:
        return np.zeros(g.shape, bool)
    m = th > threshold_otsu(th)
    try:                                    # scikit-image >= 0.26
        return morphology.remove_small_objects(m, max_size=min_size - 1)
    except TypeError:
        return morphology.remove_small_objects(m, min_size=min_size)


def optic_disc_center(rgb: np.ndarray) -> tuple[float, float]:
    """Brightest smoothed region, normalised (y, x) in [0, 1]."""
    g = rgb.astype(np.float32).mean(2)
    s = ndimage.gaussian_filter(g, sigma=max(g.shape) / 25)
    y, x = np.unravel_index(np.argmax(s), s.shape)
    return y / g.shape[0], x / g.shape[1]


# ============================================================ vessel graph and paths
class VesselGraph(Data):
    """Graph with root-to-leaf paths. ``paths`` holds local node indices (-1 = padding);
    ``path_batch`` is the graph index of each path after batching."""

    def __inc__(self, key, value, *args, **kwargs):
        if key == "paths":
            return 0
        if key == "path_batch":
            return 1
        return super().__inc__(key, value, *args, **kwargs)


def skeleton_graph(mask: np.ndarray, od: tuple[float, float] = (0.5, 0.5), max_nodes: int = 256,
                   max_paths: int = 32, path_len: int = 16) -> VesselGraph:
    """Topological vessel graph rooted at the optic disc.

    Nodes: skeleton end-points (degree 1) and junctions (degree >= 3); one node per loop
    without such points. Edges: skeleton segments, with length and tortuosity
    (length / chord). Paths: root-to-leaf node sequences of the BFS tree of each
    component, rooted at the node closest to the optic disc, longest first.
    """
    H, W = mask.shape
    skel = skeletonize(mask.astype(bool))
    ys, xs = np.nonzero(skel)
    if len(ys) < 2:
        return VesselGraph(x=torch.zeros(1, NODE_FEATS), pos=torch.full((1, 2), 0.5),
                           edge_index=torch.zeros(2, 0, dtype=torch.long), edge_attr=torch.zeros(0, EDGE_FEATS),
                           paths=torch.zeros(1, path_len, dtype=torch.long).fill_(-1).index_fill_(1, torch.tensor([0]), 0),
                           path_batch=torch.zeros(1, dtype=torch.long))

    pix = {(y, x): i for i, (y, x) in enumerate(zip(ys.tolist(), xs.tolist()))}
    nbrs = [[pix[(y + dy, x + dx)] for dy, dx in _OFFS if (y + dy, x + dx) in pix]
            for y, x in zip(ys.tolist(), xs.tolist())]
    deg = np.array([len(n) for n in nbrs])
    is_key = deg != 2
    seen = np.zeros(len(ys), bool)
    for s in range(len(ys)):                       # loops without key points get one
        if seen[s]:
            continue
        comp, q, has_key = [], deque([s]), False
        seen[s] = True
        while q:
            u = q.popleft()
            comp.append(u)
            has_key |= bool(is_key[u])
            for v in nbrs[u]:
                if not seen[v]:
                    seen[v] = True
                    q.append(v)
        if not has_key:
            is_key[comp[0]] = True
    key_ids = np.flatnonzero(is_key)
    node_of = {int(p): k for k, p in enumerate(key_ids)}

    edges, used = {}, set()
    for p in key_ids.tolist():
        for first in nbrs[p]:
            if (p, first) in used:
                continue
            prev, cur, length = p, first, 1
            used.add((p, first))
            while not is_key[cur]:
                nxt = [v for v in nbrs[cur] if v != prev]
                if not nxt:
                    break
                prev, cur = cur, nxt[0]
                length += 1
            used.add((cur, prev))
            if is_key[cur]:
                a, b = node_of[p], node_of[cur]
                if a != b:
                    chord = max(np.hypot(ys[p] - ys[cur], xs[p] - xs[cur]), 1.0)
                    k = (min(a, b), max(a, b))
                    if k not in edges or length < edges[k][0]:
                        edges[k] = (length, length / chord)

    n = len(key_ids)
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    comp_id = -np.ones(n, int)
    comps = []
    for s in range(n):                             # connected components of the key-point graph
        if comp_id[s] >= 0:
            continue
        q, members = deque([s]), [s]
        comp_id[s] = len(comps)
        while q:
            u = q.popleft()
            for v in adj[u]:
                if comp_id[v] < 0:
                    comp_id[v] = len(comps)
                    members.append(v)
                    q.append(v)
        comps.append(members)
    comps.sort(key=len, reverse=True)
    keep, total = [], 0
    for c in comps:                                # keep the largest components
        if total + len(c) > max_nodes:
            break
        keep += c
        total += len(c)
    if not keep:
        keep = comps[0][:max_nodes]
    keep = sorted(keep)
    remap = -np.ones(n, int)
    remap[keep] = np.arange(len(keep))

    py, px = ys[key_ids[keep]] / H, xs[key_ids[keep]] / W
    d_od = np.hypot(py - od[0], px - od[1]) / np.sqrt(2)
    kdeg = deg[key_ids[keep]]
    roots = set()
    paths = []
    sub_adj = [[remap[v] for v in adj[u] if remap[v] >= 0] for u in keep]
    seen = np.zeros(len(keep), bool)
    for s in range(len(keep)):                     # BFS tree of each kept component, rooted near the disc
        if seen[s]:
            continue
        q, members = deque([s]), [s]
        seen[s] = True
        while q:
            u = q.popleft()
            for v in sub_adj[u]:
                if not seen[v]:
                    seen[v] = True
                    members.append(v)
                    q.append(v)
        root = min(members, key=lambda i: d_od[i])
        roots.add(root)
        parent = {root: -1}
        order, q = [root], deque([root])
        while q:
            u = q.popleft()
            for v in sub_adj[u]:
                if v not in parent:
                    parent[v] = u
                    order.append(v)
                    q.append(v)
        children = {u: 0 for u in order}
        for v, u in parent.items():
            if u >= 0:
                children[u] += 1
        leaves = [u for u in order if children[u] == 0] or [root]
        for leaf in leaves:
            path, u = [], leaf
            while u != -1:
                path.append(u)
                u = parent[u]
            paths.append(path[::-1])               # root -> leaf (disc -> periphery)
    paths.sort(key=len, reverse=True)
    paths = paths[:max_paths]
    P = -torch.ones(len(paths), path_len, dtype=torch.long)
    for i, p in enumerate(paths):
        if len(p) > path_len:                      # uniform resampling along the path, ends kept
            p = [p[j] for j in np.linspace(0, len(p) - 1, path_len).round().astype(int)]
        P[i, :len(p)] = torch.tensor(p)

    feats = np.stack([py, px, np.minimum(kdeg, 4) / 4.0, (kdeg == 1).astype(float), (kdeg >= 3).astype(float),
                      d_od, np.isin(np.arange(len(keep)), list(roots)).astype(float)], 1)
    src, dst, attr = [], [], []
    for (a, b), (L, tort) in edges.items():
        if remap[a] >= 0 and remap[b] >= 0:
            src += [remap[a], remap[b]]
            dst += [remap[b], remap[a]]
            attr += [[L / max(H, W), min(tort, 5.0) / 5.0]] * 2
    return VesselGraph(x=torch.tensor(feats, dtype=torch.float),
                       pos=torch.tensor(np.stack([py, px], 1), dtype=torch.float),
                       edge_index=torch.tensor([src, dst], dtype=torch.long).reshape(2, -1),
                       edge_attr=torch.tensor(attr, dtype=torch.float).reshape(-1, EDGE_FEATS),
                       paths=P, path_batch=torch.zeros(len(paths), dtype=torch.long))


def flip_graph(g: VesselGraph, horizontal: bool) -> VesselGraph:
    """Mirror node positions (and features) consistently with an image flip."""
    g = g.clone()
    j = 1 if horizontal else 0
    g.pos[:, j] = 1 - g.pos[:, j]
    g.x[:, j] = 1 - g.x[:, j]
    return g


# ============================================================ preprocessing cache (speed)
def _prep_one(args):
    path, mask_path, size = args
    from PIL import Image
    img = np.asarray(Image.open(path).convert("RGB").resize((size, size), Image.BILINEAR))
    if mask_path:
        m = np.asarray(Image.open(mask_path).convert("L").resize((size, size), Image.NEAREST)) > 127
    else:
        m = pseudo_vessel_mask(img)
    od = optic_disc_center(img)
    g = skeleton_graph(m, od)
    return img, np.packbits(m), od, {k: v for k, v in g.to_dict().items()}


def build_cache(items, cache_dir, size=300, workers=None):
    """Pre-compute every image once: resized pixels (uint8 memmap), vessel mask (bit-packed),
    optic disc, vessel graph and paths. ``items``: list of (image_path, mask_path or None)."""
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    N = len(items)
    imgs = np.lib.format.open_memmap(cache / "images.npy", mode="w+", dtype=np.uint8, shape=(N, size, size, 3))
    masks = np.lib.format.open_memmap(cache / "masks.npy", mode="w+", dtype=np.uint8,
                                      shape=(N, (size * size + 7) // 8))
    ods, graphs = np.zeros((N, 2), np.float32), []
    with Pool(workers or os.cpu_count()) as pool:
        for i, (img, m, od, g) in enumerate(pool.imap(_prep_one, [(p, mp, size) for p, mp in items], chunksize=8)):
            imgs[i], masks[i], ods[i] = img, m, od
            graphs.append(g)
            if (i + 1) % 1000 == 0:
                print(f"  cached {i + 1}/{N}", flush=True)
    imgs.flush()
    masks.flush()
    np.save(cache / "optic_disc.npy", ods)
    torch.save(graphs, cache / "graphs.pt")
    json.dump({"size": size, "n": N, "images": [p for p, _ in items]}, open(cache / "index.json", "w"))
    return cache


class CachedFundus(torch.utils.data.Dataset):
    """Reads the cache; returns uint8 image (C, S, S), grade, mask (1, S, S) and graph.
    Random horizontal/vertical flips are applied to the image, mask and graph together."""

    def __init__(self, cache_dir, indices, grades, augment=False, need_mask=True, need_graph=True):
        self.cache, self.idx, self.grades = Path(cache_dir), np.asarray(indices), np.asarray(grades)
        self.augment, self.need_mask, self.need_graph = augment, need_mask, need_graph
        meta = json.load(open(self.cache / "index.json"))
        self.size = meta["size"]
        self.imgs = self.masks = self.graphs = None

    def _open(self):
        self.imgs = np.load(self.cache / "images.npy", mmap_mode="r")
        self.masks = np.load(self.cache / "masks.npy", mmap_mode="r")
        self.graphs = torch.load(self.cache / "graphs.pt", weights_only=False) if self.need_graph else None

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        if self.imgs is None:
            self._open()
        i = int(self.idx[k])
        img = np.array(self.imgs[i])
        S = self.size
        m = (np.unpackbits(self.masks[i])[: S * S].reshape(S, S) if self.need_mask
             else np.zeros((1, 1), np.uint8))
        g = VesselGraph(**self.graphs[i]) if self.need_graph else None
        if self.augment:
            if np.random.rand() < 0.5:
                img, m = img[:, ::-1], (m[:, ::-1] if self.need_mask else m)
                g = flip_graph(g, True) if g is not None else None
            if np.random.rand() < 0.5:
                img, m = img[::-1], (m[::-1] if self.need_mask else m)
                g = flip_graph(g, False) if g is not None else None
        x = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1)
        mask = torch.from_numpy(np.ascontiguousarray(m)).float()[None]
        return x, int(self.grades[k]), mask, g


def collate(batch):
    xs, ys, ms, gs = zip(*batch)
    graphs = Batch.from_data_list(list(gs)) if gs[0] is not None else None
    return torch.stack(xs), torch.tensor(ys), torch.stack(ms), graphs


def gpu_preprocess(x_uint8, size, device):
    """uint8 (B, 3, S, S) -> normalised float on the GPU, resized to the model input size."""
    x = x_uint8.to(device, non_blocking=True).float().div_(255)
    if x.shape[-1] != size:
        x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False, antialias=True)
    mean = torch.tensor(IMAGENET_MEAN, device=device)[:, None, None]
    std = torch.tensor(IMAGENET_STD, device=device)[:, None, None]
    return ((x - mean) / std).contiguous(memory_format=torch.channels_last)


# ============================================================ networks
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    """Small U-Net with skip connections (auxiliary vessel segmentation)."""

    def __init__(self, c=(16, 32, 64)):
        super().__init__()
        self.e1, self.e2, self.b = ConvBlock(3, c[0]), ConvBlock(c[0], c[1]), ConvBlock(c[1], c[2])
        self.d2, self.d1 = ConvBlock(c[2] + c[1], c[1]), ConvBlock(c[1] + c[0], c[0])
        self.out = nn.Conv2d(c[0], 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        b = self.b(F.max_pool2d(e2, 2))
        d2 = self.d2(torch.cat([F.interpolate(b, size=e2.shape[-2:], mode="bilinear", align_corners=False), e2], 1))
        d1 = self.d1(torch.cat([F.interpolate(d2, size=e1.shape[-2:], mode="bilinear", align_corners=False), e1], 1))
        return self.out(d1)


class CoralHead(nn.Module):
    """K-1 rank logits sharing one weight vector (rank-consistent, Cao et al. 2020)."""

    def __init__(self, dim, num_classes=NUM_CLASSES):
        super().__init__()
        self.w = nn.Linear(dim, 1, bias=False)
        self.b = nn.Parameter(torch.zeros(num_classes - 1))

    def forward(self, h):
        return self.w(h) + self.b


class TAGMTLDR(nn.Module):
    def __init__(self, pretrained=True, dim=256, heads=4, head="coral", backbone="efficientnet_b3",
                 use_graph=True, use_paths=True, lesion_aware=True, fusion="cross", use_seg=True):
        """Ablation switches: use_graph (vessel graph, GAT), use_paths (vessel-path GRU),
        lesion_aware (CNN features sampled at graph nodes), fusion ('cross' | 'concat'),
        use_seg (auxiliary U-Net)."""
        super().__init__()
        import timm
        if fusion not in ("cross", "concat"):
            raise ValueError(fusion)
        self.use_graph, self.use_paths = use_graph, use_paths and use_graph
        self.lesion_aware, self.fusion, self.use_seg = lesion_aware, fusion, use_seg
        self.encoder = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="")
        c = self.encoder.num_features
        self.lesion_proj = nn.Linear(c, dim)
        self.unet = UNet() if use_seg else None
        self.node_geo = nn.Sequential(nn.Linear(NODE_FEATS, dim), nn.GELU())
        self.node_lesion = nn.Linear(c, dim)
        self.g1 = GATConv(dim, dim // heads, heads=heads, edge_dim=EDGE_FEATS)
        self.g2 = GATConv(dim, dim, heads=1, edge_dim=EDGE_FEATS)
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.gru = nn.GRU(dim, dim // 2, batch_first=True, bidirectional=True)
        self.path_score = nn.Linear(dim, 1)
        self.cross = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.fuse = nn.Sequential(nn.Linear(4 * dim, dim), nn.GELU(), nn.Dropout(0.3))
        self.head = CoralHead(dim) if head == "coral" else nn.Linear(dim, NUM_CLASSES)

    def _node_embeddings(self, fmap, g):
        h = self.node_geo(g.x)
        if self.lesion_aware:                                          # CNN evidence sampled at each node
            pos, valid = to_dense_batch(g.pos, g.batch, batch_size=fmap.size(0))
            grid = pos[..., [1, 0]].mul(2).sub(1).unsqueeze(2)         # (B, Nmax, 1, 2) as (x, y) in [-1, 1]
            samp = F.grid_sample(fmap.float(), grid.float(), align_corners=False).squeeze(-1).transpose(1, 2)
            h = h + self.node_lesion(samp[valid].to(h.dtype))
        h = F.elu(self.n1(self.g1(h, g.edge_index, g.edge_attr)))
        return self.n2(self.g2(h, g.edge_index, g.edge_attr))

    def _paths(self, nodes, g, B):
        if g.paths.numel() == 0:
            return nodes.new_zeros(0, nodes.size(1)), g.path_batch, nodes.new_zeros(B, nodes.size(1))
        offs = g.ptr[g.path_batch].unsqueeze(1)
        idx = torch.where(g.paths >= 0, g.paths + offs, torch.zeros_like(g.paths))
        seq = nodes[idx] * (g.paths >= 0).unsqueeze(-1)
        lengths = (g.paths >= 0).sum(1).clamp_min(1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(seq, lengths, batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)                                        # (2, P, d/2)
        emb = torch.cat([h[0], h[1]], 1)
        a = seg_softmax(self.path_score(emb).squeeze(-1), g.path_batch, num_nodes=B)
        pooled = torch.zeros(B, emb.size(1), device=emb.device, dtype=emb.dtype).index_add_(
            0, g.path_batch, emb * a.unsqueeze(-1))
        return emb, g.path_batch, pooled

    def forward(self, x, graphs: Batch | None = None):
        fmap = self.encoder.forward_features(x)                        # (B, C, h, w)
        tokens = self.lesion_proj(fmap.flatten(2).transpose(1, 2))      # (B, h*w, d)
        B, d = x.size(0), tokens.size(-1)
        lesion_global = tokens.mean(1)
        vessel_logits = self.unet(x) if self.use_seg else None
        zeros = tokens.new_zeros(B, d)
        attn, graph_global, path_global, fused = None, zeros, zeros, lesion_global
        if self.use_graph:
            g = graphs.to(x.device)
            nodes = self._node_embeddings(fmap, g).to(tokens.dtype)
            graph_global = global_mean_pool(nodes, g.batch, size=B)
            keys, valid = to_dense_batch(nodes, g.batch, batch_size=B)
            if self.use_paths:
                pe, pb, path_global = self._paths(nodes, g, B)
                if pe.numel():
                    pk, pv = to_dense_batch(pe.to(tokens.dtype), pb, batch_size=B)
                    keys, valid = torch.cat([keys, pk], 1), torch.cat([valid, pv], 1)
            if self.fusion == "cross":
                attended, attn = self.cross(tokens, keys, keys, key_padding_mask=~valid)
                fused = self.norm(tokens + attended).mean(1)           # residual keeps lesion information
        h = self.fuse(torch.cat([fused, lesion_global, graph_global, path_global.to(tokens.dtype)], 1))
        return self.head(h), vessel_logits, attn


# ============================================================ losses and metrics
def coral_loss(logits, y):
    levels = (y.unsqueeze(1) > torch.arange(NUM_CLASSES - 1, device=y.device)).float()
    return F.binary_cross_entropy_with_logits(logits, levels)


def ce_mse_loss(logits, y, w=0.5):
    probs = logits.softmax(1)
    expected = (probs * torch.arange(NUM_CLASSES, device=y.device)).sum(1)
    return F.cross_entropy(logits, y) + w * F.mse_loss(expected, y.float())


def seg_loss(vessel_logits, target):
    if target.shape[-1] != vessel_logits.shape[-1]:
        target = F.interpolate(target, size=vessel_logits.shape[-2:], mode="nearest")
    bce = F.binary_cross_entropy_with_logits(vessel_logits, target)
    p = torch.sigmoid(vessel_logits)
    dice = 1 - (2 * (p * target).sum((1, 2, 3)) + 1) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)
    return bce + dice.mean()


def class_probs(logits, head, T=1.0):
    """(B, 5) class probabilities for every head type, with temperature T."""
    logits = logits / T
    if head == "coral":
        s = torch.cummin(torch.sigmoid(logits), dim=1).values           # P(y > k), monotone
        ones, zeros = torch.ones_like(s[:, :1]), torch.zeros_like(s[:, :1])
        cum = torch.cat([ones, s, zeros], 1)
        return (cum[:, :-1] - cum[:, 1:]).clamp_min(0)
    return logits.softmax(1)


def metrics(y_true, y_pred) -> dict:
    from sklearn.metrics import accuracy_score, cohen_kappa_score
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    nr = lambda g: float((y_pred[y_true == g] <= 1).mean()) if (y_true == g).any() else float("nan")
    return {"accuracy": accuracy_score(y_true, y_pred),
            "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
            "sev_nr": nr(3), "pdr_nr": nr(4),
            "grade_0_4_confusions": int(((y_true == 0) & (y_pred == 4)).sum() + ((y_true == 4) & (y_pred == 0)).sum())}


def patient_folds(grades, patients, n_splits=5, seed=42):
    """Patient-grouped stratified folds: the two eyes of a patient stay together."""
    from sklearn.model_selection import StratifiedGroupKFold
    return list(StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed)
                .split(np.zeros(len(grades)), grades, groups=patients))


# ============================================================ smoke test
def _smoke():
    import time
    torch.manual_seed(0)
    ring = np.zeros((64, 64), bool)
    yy, xx = np.ogrid[:64, :64]
    r = np.sqrt((yy - 32) ** 2 + (xx - 32) ** 2)
    ring[(r > 18) & (r < 21)] = True
    g = skeleton_graph(ring)
    assert g.x.shape[1] == NODE_FEATS and g.num_nodes >= 1, "ring"
    tee = np.zeros((64, 64), bool)
    tee[10:13, 5:60] = True
    tee[10:55, 31:34] = True
    g = skeleton_graph(tee, od=(0.0, 0.0))
    ends, junc = int(g.x[:, 3].sum()), int(g.x[:, 4].sum())
    assert ends >= 3 and junc >= 1, f"T shape: {ends} ends, {junc} junctions"
    assert g.paths.shape[0] >= 2 and (g.paths[:, 0] == g.paths[0, 0]).all(), "paths share the disc root"
    assert g.edge_attr.shape[1] == EDGE_FEATS
    e = skeleton_graph(np.zeros((32, 32), bool))
    assert e.edge_index.shape == (2, 0) and e.paths.shape[1] == 16
    f = flip_graph(g, True)
    assert torch.allclose(f.pos[:, 1], 1 - g.pos[:, 1])
    b = Batch.from_data_list([g, skeleton_graph(ring), e])
    assert b.path_batch.max().item() == 2 and b.paths.max().item() < max(g.num_nodes, 1) + 64

    # timing of the per-image CPU work removed from the training loop by the cache
    rgb = (np.random.rand(300, 300, 3) * 255).astype(np.uint8)
    t0 = time.time()
    for _ in range(3):
        skeleton_graph(pseudo_vessel_mask(rgb), optic_disc_center(rgb))
    cpu_ms = (time.time() - t0) / 3 * 1000

    for flags in ({}, {"use_paths": False}, {"lesion_aware": False}, {"fusion": "concat"}, {"use_graph": False},
                  {"use_seg": False}):
        model = TAGMTLDR(pretrained=False, **flags)
        opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
        x = torch.randn(3, 3, 128, 128)
        y = torch.tensor([1, 3, 4])
        m = (torch.rand(3, 1, 128, 128) > 0.9).float()
        logits, vlog, attn = model(x, b)
        loss = coral_loss(logits, y) + (0.5 * seg_loss(vlog, m) if vlog is not None else 0)
        opt.zero_grad()
        loss.backward()
        assert model.encoder.conv_stem.weight.grad.abs().sum() > 0
        if flags.get("use_graph", True):
            assert model.g1.lin.weight.grad.abs().sum() > 0 if hasattr(model.g1, "lin") else True
            if flags.get("use_paths", True):
                assert model.gru.weight_ih_l0.grad.abs().sum() > 0, "GRU receives gradient"
            if flags.get("lesion_aware", True):
                assert model.node_lesion.weight.grad.abs().sum() > 0, "lesion-aware nodes receive gradient"
        if vlog is not None:
            assert model.unet.out.weight.grad.abs().sum() > 0
        assert logits.shape == (3, NUM_CLASSES - 1)
        P = class_probs(logits.detach(), "coral", T=1.5)
        assert torch.allclose(P.sum(1), torch.ones(3), atol=1e-5)
        print("  ok", flags or "full model")
    print(f"smoke test passed (per-image CPU graph extraction cached once: about {cpu_ms:.0f} ms/image saved per epoch)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    if ap.parse_args().smoke:
        _smoke()

In [ ]:
%%writefile benchmark.py
"""Benchmark of TAG-MTL-DR v2 against reproduced baselines under one protocol.

Every method -- baselines, the proposed model and its ablations -- is trained and
evaluated on the same patient-grouped folds, seeds, cache, budget and external
datasets. Per-image predictions (temperature-calibrated on the validation patients)
are written to ``predictions.csv``; ``analysis.py`` turns them into the paper tables.

Speed: one pre-processing cache shared by all methods (``build_cache``), GPU resizing
and normalisation, mixed precision, channels-last memory, persistent workers.

Manifest (CSV): ``image,grade,patient,dataset,role[,mask]`` (role = internal | external)

Usage
-----
    python benchmark.py --manifest manifest.csv --cache cache/ --out runs/main \
        --methods all --seeds 42 43 44 --folds 5 --epochs 30
    python analysis.py --runs runs/main
"""

from __future__ import annotations

import argparse
import csv
import json
import platform
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from tag_mtl_dr import (NUM_CLASSES, CachedFundus, CoralHead, TAGMTLDR, build_cache, ce_mse_loss, class_probs,
                        collate, coral_loss, gpu_preprocess, metrics, patient_folds, seg_loss)

# kind: "cnn" = timm backbone + head; "tag" = TAG-MTL-DR v2 with ablation switches
METHODS = {
    "ResNet-50":               {"kind": "cnn", "backbone": "resnet50", "head": "ce", "size": 300},
    "DenseNet-121":            {"kind": "cnn", "backbone": "densenet121", "head": "ce", "size": 300},
    "EfficientNet-B3":         {"kind": "cnn", "backbone": "efficientnet_b3", "head": "ce", "size": 300},
    "EfficientNet-B3 + CORAL": {"kind": "cnn", "backbone": "efficientnet_b3", "head": "coral", "size": 300},
    "ConvNeXt-T":              {"kind": "cnn", "backbone": "convnext_tiny", "head": "ce", "size": 288},
    "ViT-B/16":                {"kind": "cnn", "backbone": "vit_base_patch16_224", "head": "ce", "size": 224},
    "Swin-T":                  {"kind": "cnn", "backbone": "swin_tiny_patch4_window7_224", "head": "ce", "size": 224},
    "TAG-MTL-DR (ours)":       {"kind": "tag", "size": 300},
    "w/o vessel-path GRU":     {"kind": "tag", "size": 300, "use_paths": False},
    "w/o lesion-aware nodes":  {"kind": "tag", "size": 300, "lesion_aware": False},
    "w/o vessel graph":        {"kind": "tag", "size": 300, "use_graph": False},
    "w/o segmentation task":   {"kind": "tag", "size": 300, "use_seg": False},
    "concat instead of cross-attention": {"kind": "tag", "size": 300, "fusion": "concat"},
    "CE + MSE instead of CORAL": {"kind": "tag", "size": 300, "head": "ce_mse"},
}
BASELINES = ["ResNet-50", "DenseNet-121", "EfficientNet-B3", "EfficientNet-B3 + CORAL",
             "ConvNeXt-T", "ViT-B/16", "Swin-T"]
ABLATIONS = ["w/o vessel-path GRU", "w/o lesion-aware nodes", "w/o vessel graph", "w/o segmentation task",
             "concat instead of cross-attention", "CE + MSE instead of CORAL"]


class CNNClassifier(nn.Module):
    def __init__(self, backbone, head, pretrained):
        super().__init__()
        import timm
        self.encoder = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        d = self.encoder.num_features
        self.drop = nn.Dropout(0.3)
        self.head = CoralHead(d) if head == "coral" else nn.Linear(d, NUM_CLASSES)

    def forward(self, x, graphs=None):
        return self.head(self.drop(self.encoder(x))), None, None


def head_of(spec):
    return spec.get("head", "coral")


def build(spec, pretrained, backbone_override=None):
    if spec["kind"] == "cnn":
        return CNNClassifier(backbone_override or spec["backbone"], spec["head"], pretrained)
    return TAGMTLDR(pretrained=pretrained, head="coral" if head_of(spec) == "coral" else "linear",
                    backbone=backbone_override or "efficientnet_b3",
                    use_graph=spec.get("use_graph", True), use_paths=spec.get("use_paths", True),
                    lesion_aware=spec.get("lesion_aware", True), fusion=spec.get("fusion", "cross"),
                    use_seg=spec.get("use_seg", True))


def loss_fn(spec, logits, y, vlog, m):
    head = head_of(spec)
    loss = (coral_loss(logits, y) if head == "coral" else ce_mse_loss(logits, y) if head == "ce_mse"
            else F.cross_entropy(logits, y))
    if vlog is not None:
        loss = loss + spec.get("lam_seg", 0.5) * seg_loss(vlog, m)
    return loss


def read_manifest(path):
    rows = list(csv.DictReader(open(path, newline="")))
    need = {"image", "grade", "patient", "dataset", "role"}
    if not rows or not need <= set(rows[0]):
        raise ValueError(f"manifest needs the columns {sorted(need)}")
    for r in rows:
        r["grade"] = int(r["grade"])
        if r["role"] not in ("internal", "external"):
            raise ValueError(f"role must be internal|external: {r}")
    return rows


def ensure_cache(rows, cache_dir, size, workers):
    cache = Path(cache_dir)
    meta = cache / "index.json"
    if meta.exists():
        m = json.load(open(meta))
        if m["images"] == [r["image"] for r in rows] and m["size"] == size:
            return cache
        raise SystemExit(f"{cache} was built for another manifest/size; use a new --cache folder")
    print(f"building the pre-processing cache in {cache} (once for all methods) ...", flush=True)
    t0 = time.time()
    build_cache([(r["image"], r.get("mask") or None) for r in rows], cache, size, workers)
    print(f"cache built in {time.time() - t0:.0f} s", flush=True)
    return cache


def loader(cache, rows, idx, spec, train, args, seed):
    tag = spec["kind"] == "tag"
    ds = CachedFundus(cache, idx, [rows[i]["grade"] for i in idx], augment=train,
                      need_mask=tag and spec.get("use_seg", True), need_graph=tag and spec.get("use_graph", True))
    sampler = None
    if train:                                                       # class-balanced sampling
        g = np.array([rows[i]["grade"] for i in idx])
        w = 1.0 / np.maximum(np.bincount(g, minlength=NUM_CLASSES), 1)[g]
        sampler = torch.utils.data.WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), len(idx),
                                                         generator=torch.Generator().manual_seed(seed))
    return torch.utils.data.DataLoader(ds, batch_size=args.batch, sampler=sampler, shuffle=False,
                                       num_workers=args.workers, persistent_workers=args.workers > 0,
                                       pin_memory=torch.cuda.is_available(), collate_fn=collate,
                                       drop_last=train and len(idx) > args.batch)


@torch.no_grad()
def predict_logits(model, dl, spec, device, size):
    model.eval()
    L, n, t = [], 0, 0.0
    for x, _, _, g in dl:
        x = gpu_preprocess(x, size, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            logits, _, _ = model(x, g)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t += time.time() - t0
        n += x.size(0)
        L.append(logits.float().cpu())
    return torch.cat(L), 1000 * t / max(n, 1)


def fit_temperature(logits, y, head):
    """Temperature minimising the validation negative log-likelihood (grid search)."""
    y = torch.as_tensor(y)
    best_T, best = 1.0, float("inf")
    for T in np.linspace(0.5, 3.0, 26):
        P = class_probs(logits, head, float(T)).clamp_min(1e-7)
        nll = -torch.log(P[torch.arange(len(y)), y]).mean().item()
        if nll < best:
            best, best_T = nll, float(T)
    return best_T


def run(cache, rows, spec, train_idx, val_idx, eval_sets, args, seed, device):
    torch.manual_seed(seed)
    np.random.seed(seed)
    size = args.size_override or spec["size"]
    model = build(spec, not args.no_pretrained, args.backbone_override).to(device).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    tr = loader(cache, rows, train_idx, spec, True, args, seed)
    va = loader(cache, rows, val_idx, spec, False, args, seed)
    yv = np.array([rows[i]["grade"] for i in val_idx])
    best, best_state, wait, t_train = -np.inf, None, 0, time.time()
    for epoch in range(args.epochs):
        model.train()
        for x, y, m, g in tr:
            x, y = gpu_preprocess(x, size, device), y.to(device)
            m = m.to(device, non_blocking=True)
            with torch.autocast(device.type, enabled=device.type == "cuda"):
                logits, vlog, _ = model(x, g)
            loss = loss_fn(spec, logits.float(), y, None if vlog is None else vlog.float(), m)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sched.step()
        lv, _ = predict_logits(model, va, spec, device, size)
        q = metrics(yv, class_probs(lv, head_of(spec)).argmax(1).numpy())["qwk"]
        if q > best + 1e-4:
            best, wait = q, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= args.patience:
                break
    t_train = time.time() - t_train
    model.load_state_dict(best_state)
    if args.save_models:                                            # for explain.py and later inference
        ck = Path(args.out) / "checkpoints"
        ck.mkdir(exist_ok=True)
        torch.save(model.state_dict(), ck / f"{spec['name'].replace('/', '_').replace(' ', '_')}_s{seed}.pt")
    lv, _ = predict_logits(model, va, spec, device, size)
    T = fit_temperature(lv, yv, head_of(spec))                     # calibration on validation patients only
    out, ms = {}, []
    for split, idx in eval_sets.items():
        L, ms_img = predict_logits(model, loader(cache, rows, idx, spec, False, args, seed), spec, device, size)
        out[split] = (idx, class_probs(L, head_of(spec), T).numpy())
        ms.append(ms_img)
    info = {"epochs": epoch + 1, "best_val_qwk": float(best), "temperature": T,
            "params": int(sum(p.numel() for p in model.parameters())),
            "train_seconds": round(t_train, 1), "infer_ms_per_image": float(np.mean(ms))}
    return out, info


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--cache", default="cache")
    ap.add_argument("--out", required=True)
    ap.add_argument("--methods", nargs="+", default=["all"], help="'all', 'baselines', 'ablations' or names")
    ap.add_argument("--seeds", nargs="+", type=int, default=[42, 43, 44])
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--patience", type=int, default=6)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--cache_size", type=int, default=300)
    ap.add_argument("--train_fraction", type=float, default=1.0,
                    help="use a patient-level fraction of the training data (data-efficiency curve)")
    ap.add_argument("--save_models", action="store_true", help="save the best checkpoint of every run")
    ap.add_argument("--no_pretrained", action="store_true")
    ap.add_argument("--backbone_override", default=None, help="smoke tests only")
    ap.add_argument("--size_override", type=int, default=None, help="smoke tests only")
    args = ap.parse_args()

    names = []
    for m in args.methods:
        names += (list(METHODS) if m == "all" else BASELINES if m == "baselines"
                  else ABLATIONS if m == "ablations" else [m])
    unknown = [n for n in names if n not in METHODS]
    if unknown:
        raise SystemExit(f"unknown methods: {unknown}; choose from {list(METHODS)}")
    names = list(dict.fromkeys(names))

    rows = read_manifest(args.manifest)
    cache = ensure_cache(rows, args.cache, args.cache_size, args.workers)
    internal = np.array([i for i, r in enumerate(rows) if r["role"] == "internal"])
    externals = sorted({r["dataset"] for r in rows if r["role"] == "external"})
    grades = np.array([rows[i]["grade"] for i in internal])
    patients = np.array([rows[i]["patient"] for i in internal])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    import sklearn, timm, torch_geometric
    meta = {"args": vars(args), "methods": {n: METHODS[n] for n in names}, "runs": [],
            "n_internal": int(len(internal)), "n_internal_patients": int(len(set(patients))),
            "externals": externals,
            "environment": {"python": platform.python_version(), "torch": torch.__version__,
                            "timm": timm.__version__, "torch_geometric": torch_geometric.__version__,
                            "sklearn": sklearn.__version__,
                            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}}

    fpred = open(out / "predictions.csv", "w", newline="")
    w = csv.writer(fpred)
    w.writerow(["method", "seed", "fold", "split", "dataset", "image", "patient", "y", "pred",
                "p0", "p1", "p2", "p3", "p4", "n_train", "n_test"])
    from sklearn.model_selection import GroupShuffleSplit
    for seed in args.seeds:
        for k, (tv_rel, te_rel) in enumerate(patient_folds(grades, patients, args.folds, seed), start=1):
            tv, te = internal[tv_rel], internal[te_rel]
            gss = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=seed + k)
            tr_rel, va_rel = next(gss.split(tv, groups=[rows[i]["patient"] for i in tv]))
            tr, va = tv[tr_rel], tv[va_rel]
            if args.train_fraction < 1.0:                           # patient-level subsampling
                tp = np.array(sorted({rows[i]["patient"] for i in tr}))
                keep = set(np.random.RandomState(seed + k).choice(tp, max(1, int(round(args.train_fraction * len(tp)))),
                                                                   replace=False))
                tr = np.array([i for i in tr if rows[i]["patient"] in keep])
            pt, pv, pe = ({rows[i]["patient"] for i in s} for s in (tr, va, te))
            assert not (pt & pe) and not (pv & pe) and not (pt & pv), "patient leakage"
            eval_sets = {"internal": te}
            for ds in externals:
                eval_sets[f"external:{ds}"] = np.array([i for i, r in enumerate(rows)
                                                        if r["role"] == "external" and r["dataset"] == ds])
            for name in names:
                spec = dict(METHODS[name], name=name)
                preds, info = run(cache, rows, spec, tr, va, eval_sets, args, seed * 100 + k, device)
                for split, (idx, P) in preds.items():
                    for i, p in zip(idx, P):
                        r = rows[i]
                        w.writerow([name, seed, k, split.split(":")[0], r["dataset"], r["image"], r["patient"],
                                    r["grade"], int(p.argmax()), *[f"{v:.6f}" for v in p], len(tr), len(te)])
                fpred.flush()
                info.update({"method": name, "seed": seed, "fold": k})
                meta["runs"].append(info)
                print(f"seed {seed} fold {k} {name:<36} val QWK {info['best_val_qwk']:.4f}  T={info['temperature']:.2f}  "
                      f"{info['train_seconds']} s  {info['infer_ms_per_image']:.1f} ms/img", flush=True)
                json.dump(meta, open(out / "runs.json", "w"), indent=1)
    fpred.close()
    print(f"wrote {out / 'predictions.csv'} and {out / 'runs.json'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analysis.py
"""Paper tables, figures and statistics from ``benchmark.py`` predictions.

Outputs (in ``<runs>/paper``):
  table_sota.tex        reproduced baselines vs the proposed model, same protocol
  table_ablation.tex    ablations of the proposed model
  table_external.tex    external validation (models trained on internal data only)
  table_pergrade.tex    per-grade precision / recall / F1 of the proposed model
  table_literature.tex  results *reported* in the literature (from a sourced CSV),
                        printed separately because the protocols differ
  results_macros.tex    \\newcommand macros with every number quoted in the text
  fig_*.pdf             QWK with CIs, confusion matrix, reliability diagram
  results.json          everything above in machine-readable form

Statistics: mean +/- SD over seed x fold runs; 95% CIs by patient-level bootstrap;
comparison with the reference model by the corrected repeated k-fold t-test
(Bouckaert & Frank 2004, Nadeau & Bengio 2003) with Holm correction; external sets:
exact McNemar test and paired patient bootstrap of the QWK difference, using the
ensemble of the cross-validated models; Sev-NR / PDR-NR with exact Clopper-Pearson
intervals; ECE (15 bins) and Brier score.
"""

from __future__ import annotations

import argparse
import csv
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy import stats
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)

GRADES = ["No DR", "Mild", "Moderate", "Severe", "PDR"]


# ----------------------------------------------------------------------------- metrics
def ece(y, P, bins=15):
    conf, pred = P.max(1), P.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)


def clopper_pearson(k, n, a=0.05):
    if n == 0:
        return (float("nan"), float("nan"))
    lo = stats.beta.ppf(a / 2, k, n - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(1 - a / 2, k + 1, n - k) if k < n else 1.0
    return float(lo), float(hi)


def all_metrics(y, P):
    pred = P.argmax(1)
    out = {"accuracy": accuracy_score(y, pred),
           "qwk": cohen_kappa_score(y, pred, weights="quadratic"),
           "macro_f1": f1_score(y, pred, average="macro", labels=range(5), zero_division=0),
           "ece": ece(y, P), "brier": float(((P - np.eye(5)[y]) ** 2).sum(1).mean())}
    ref = (y >= 2).astype(int)
    out["auc_referable"] = roc_auc_score(ref, P[:, 2:].sum(1)) if 0 < ref.sum() < len(ref) else float("nan")
    for g, key in [(3, "sev"), (4, "pdr")]:
        m = y == g
        k, n = int((pred[m] <= 1).sum()), int(m.sum())
        out[f"{key}_nr"] = k / n if n else float("nan")
        out[f"{key}_nr_ci"] = clopper_pearson(k, n)
        out[f"{key}_n"] = n
    out["conf_0_4"] = int(((y == 0) & (pred == 4)).sum() + ((y == 4) & (pred == 0)).sum())
    return {k: (float(v) if isinstance(v, (np.floating, float, int, np.integer)) and k not in ("sev_n", "pdr_n", "conf_0_4") else v)
            for k, v in out.items()}


def patient_bootstrap(y, P, patients, fn, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    uniq, inv = np.unique(patients, return_inverse=True)
    groups = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))])
        try:
            vals.append(fn(y[idx], P[idx]))
        except ValueError:
            continue
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def qwk_of(y, P):
    return cohen_kappa_score(y, P.argmax(1), weights="quadratic")


def corrected_repeated_t(d, ratio):
    n = len(d)
    var = np.var(d, ddof=1)
    if n < 2 or var == 0:
        return float("nan"), float("nan")
    t = d.mean() / np.sqrt((1 / n + ratio) * var)
    return float(t), float(2 * stats.t.sf(abs(t), n - 1))


def holm(pvals: dict) -> dict:
    items = sorted((p, k) for k, p in pvals.items() if not np.isnan(p))
    m, adj, run = len(items), {}, 0.0
    for i, (p, k) in enumerate(items):
        run = max(run, min(1.0, (m - i) * p))
        adj[k] = run
    for k, p in pvals.items():
        adj.setdefault(k, float("nan"))
    return adj


def mcnemar_exact(correct_a, correct_b):
    b = int((correct_a & ~correct_b).sum())
    c = int((~correct_a & correct_b).sum())
    return float(stats.binomtest(min(b, c), b + c, 0.5).pvalue) if b + c else 1.0


# ----------------------------------------------------------------------------- loading
def load(runs_dir):
    rows = list(csv.DictReader(open(Path(runs_dir) / "predictions.csv")))
    meta = json.load(open(Path(runs_dir) / "runs.json"))
    data = defaultdict(list)          # (method, split, dataset) -> rows
    for r in rows:
        data[(r["method"], r["split"], r["dataset"])].append(r)
    return data, meta


def arrays(rs):
    y = np.array([int(r["y"]) for r in rs])
    P = np.array([[float(r[f"p{k}"]) for k in range(5)] for r in rs])
    pat = np.array([r["patient"] for r in rs])
    img = np.array([r["image"] for r in rs])
    return y, P, pat, img


def fmt(v, d=3):
    return "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{d}f}"


def fmt_p(p):
    return "--" if np.isnan(p) else ("$<$0.001" if p < 0.001 else f"{p:.3f}")


def tex_escape(s):
    return s.replace("&", r"\&").replace("%", r"\%").replace("_", r"\_").replace("#", r"\#")


# ----------------------------------------------------------------------------- analysis
def internal_summary(data, method, ratio):
    rs = [r for (m, s, _), v in data.items() if m == method and s == "internal" for r in v]
    by_run = defaultdict(list)
    for r in rs:
        by_run[(r["seed"], r["fold"])].append(r)
    per_run = {k: all_metrics(*arrays(v)[:2]) for k, v in by_run.items()}
    keys = ["accuracy", "qwk", "macro_f1", "auc_referable", "ece", "brier"]
    summ = {k: (float(np.nanmean([m[k] for m in per_run.values()])),
                float(np.nanstd([m[k] for m in per_run.values()], ddof=1)) if len(per_run) > 1 else 0.0)
            for k in keys}
    # pooled out-of-fold predictions of the first seed: one prediction per internal image
    seeds = sorted({k[0] for k in by_run})
    pooled = [r for k, v in by_run.items() if k[0] == seeds[0] for r in v]
    y, P, pat, _ = arrays(pooled)
    pm = all_metrics(y, P)
    summ["qwk_pooled"] = pm["qwk"]                                   # point estimate matching the CI
    summ["qwk_ci"] = patient_bootstrap(y, P, pat, qwk_of)
    for key in ("sev_nr", "sev_nr_ci", "sev_n", "pdr_nr", "pdr_nr_ci", "pdr_n", "conf_0_4"):
        summ[key] = pm[key]
    summ["per_run_qwk"] = {f"{k[0]}-{k[1]}": m["qwk"] for k, m in per_run.items()}
    summ["pooled"] = (y, P)
    return summ


def external_summary(data, method, dataset):
    rs = [r for r in data.get((method, "external", dataset), [])]
    if not rs:
        return None
    by_img = defaultdict(list)
    for r in rs:
        by_img[r["image"]].append(r)
    imgs = sorted(by_img)
    y = np.array([int(by_img[i][0]["y"]) for i in imgs])
    pat = np.array([by_img[i][0]["patient"] for i in imgs])
    P = np.array([np.mean([[float(r[f"p{k}"]) for k in range(5)] for r in by_img[i]], 0) for i in imgs])
    runs = defaultdict(list)
    for r in rs:
        runs[(r["seed"], r["fold"])].append(r)
    q_runs = [qwk_of(*arrays(v)[:2]) for v in runs.values()]
    m = all_metrics(y, P)
    m.update({"qwk_ci": patient_bootstrap(y, P, pat, qwk_of), "qwk_runs_mean": float(np.mean(q_runs)),
              "qwk_runs_sd": float(np.std(q_runs, ddof=1)) if len(q_runs) > 1 else 0.0,
              "images": imgs, "y": y, "P": P, "patients": pat})
    return m


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--runs", required=True)
    ap.add_argument("--reference", default="TAG-MTL-DR (ours)")
    ap.add_argument("--literature", default=None, help="sourced CSV of results reported in the literature")
    ap.add_argument("--no_figures", action="store_true")
    a = ap.parse_args()

    from benchmark import ABLATIONS, BASELINES
    data, meta = load(a.runs)
    methods = list(meta["methods"])
    if a.reference not in methods:
        raise SystemExit(f"reference '{a.reference}' not in the runs: {methods}")
    out = Path(a.runs) / "paper"
    out.mkdir(exist_ok=True)
    rows = list(csv.DictReader(open(Path(a.runs) / "predictions.csv")))
    n_train = np.mean([float(r["n_train"]) for r in rows])
    n_test = np.mean([float(r["n_test"]) for r in rows])
    ratio = n_test / n_train
    params = defaultdict(list)
    for r in meta["runs"]:
        params[r["method"]].append(r["params"])

    S = {m: internal_summary(data, m, ratio) for m in methods}
    ref = S[a.reference]
    pvals = {}
    for m in methods:
        if m == a.reference:
            continue
        common = sorted(set(ref["per_run_qwk"]) & set(S[m]["per_run_qwk"]))
        d = np.array([ref["per_run_qwk"][k] - S[m]["per_run_qwk"][k] for k in common])
        S[m]["delta_qwk"] = float(d.mean()) if len(d) else float("nan")
        S[m]["t"], pvals[m] = corrected_repeated_t(d, ratio)
    padj = holm(pvals)

    def row(m, ours=False):
        s = S[m]
        name = tex_escape(m)
        name = r"\textbf{" + name + "}" if ours else name
        sev = f"{100 * s['sev_nr']:.1f} [{100 * s['sev_nr_ci'][0]:.1f}, {100 * s['sev_nr_ci'][1]:.1f}]"
        return (f"{name} & {np.mean(params[m]) / 1e6:.1f} & {fmt(s['accuracy'][0])}$\\pm${fmt(s['accuracy'][1])} & "
                f"{fmt(s['qwk'][0])}$\\pm${fmt(s['qwk'][1])} & {fmt(s['qwk_pooled'])} [{fmt(s['qwk_ci'][0])}, {fmt(s['qwk_ci'][1])}] & "
                f"{fmt(s['macro_f1'][0])} & {fmt(s['auc_referable'][0])} & {sev} & {fmt(s['ece'][0])} & "
                f"{'--' if ours else fmt_p(padj.get(m, float('nan')))} \\\\")

    head = (r"\begin{table*}[t]\centering\small" "\n"
            r"\caption{CAPTION}\label{LABEL}" "\n"
            r"\resizebox{\textwidth}{!}{\begin{tabular}{@{}lrcccccccc@{}}\toprule" "\n"
            r"Method & Params (M) & Accuracy & QWK & Pooled QWK [95\% CI] & Macro-F1 & AUC (ref.) & Sev-NR \% [95\% CI] & ECE & $p_{\mathrm{Holm}}$ \\ \midrule")
    foot = r"\bottomrule\end{tabular}}" "\n" r"\par\smallskip\footnotesize NOTE" "\n" r"\end{table*}"
    proto = (f"Patient-grouped {meta['args']['folds']}-fold cross-validation, seeds "
             f"{', '.join(map(str, meta['args']['seeds']))}; mean $\\pm$ SD over runs; pooled QWK: out-of-fold predictions of "
             f"the first seed, 95\\% CI by patient-level bootstrap; $p$: corrected repeated $k$-fold $t$-test "
             f"against {tex_escape(a.reference)}, Holm-adjusted. All methods trained with the same protocol.")
    sota = [m for m in BASELINES if m in S]
    with open(out / "table_sota.tex", "w") as f:
        f.write(head.replace("CAPTION", "Comparison with reproduced state-of-the-art baselines (internal data, same protocol).").replace("LABEL", "tab:sota") + "\n")
        for m in sota:
            f.write(row(m) + "\n")
        f.write(r"\midrule" "\n" + row(a.reference, ours=True) + "\n")
        f.write(foot.replace("NOTE", proto) + "\n")
    abl = [m for m in ABLATIONS if m in S]
    if abl:
        with open(out / "table_ablation.tex", "w") as f:
            f.write(head.replace("CAPTION", "Ablation of the proposed model (internal data).").replace("LABEL", "tab:ablation") + "\n")
            f.write(row(a.reference, ours=True) + "\n" + r"\midrule" "\n")
            for m in abl:
                f.write(row(m) + "\n")
            f.write(foot.replace("NOTE", proto) + "\n")

    # per-grade table of the reference model
    y, P = ref["pooled"]
    pr, rc, f1, sup = precision_recall_fscore_support(y, P.argmax(1), labels=range(5), zero_division=0)
    with open(out / "table_pergrade.tex", "w") as f:
        f.write(r"\begin{table}[t]\centering\small\caption{Per-grade results of " + tex_escape(a.reference)
                + r" (pooled out-of-fold predictions).}\label{tab:pergrade}" "\n"
                r"\begin{tabular}{@{}lcccr@{}}\toprule Grade & Precision & Recall & F1 & $N$ \\ \midrule" "\n")
        for g in range(5):
            f.write(f"{g} -- {GRADES[g]} & {pr[g]:.3f} & {rc[g]:.3f} & {f1[g]:.3f} & {int(sup[g])} \\\\\n")
        f.write(r"\bottomrule\end{tabular}\end{table}" "\n")

    # computational cost (all methods, same hardware)
    cost = defaultdict(lambda: defaultdict(list))
    for r in meta["runs"]:
        for k in ("params", "train_seconds", "infer_ms_per_image", "temperature"):
            if k in r:
                cost[r["method"]][k].append(r[k])
    with open(out / "table_cost.tex", "w") as f:
        f.write(r"\begin{table}[t]\centering\small\caption{Computational cost on "
                + tex_escape(meta.get("environment", {}).get("gpu", "the same hardware"))
                + r" (mean over runs).}\label{tab:cost}" "\n"
                r"\begin{tabular}{@{}lcccc@{}}\toprule Method & Params (M) & Training (min/run) & Inference (ms/image) & Temperature \\ \midrule" "\n")
        for m in sota + [a.reference] + abl:
            c = cost[m]
            f.write(f"{tex_escape(m)} & {np.mean(c['params']) / 1e6:.1f} & "
                    f"{(np.mean(c['train_seconds']) / 60 if c['train_seconds'] else float('nan')):.1f} & "
                    f"{(np.mean(c['infer_ms_per_image']) if c['infer_ms_per_image'] else float('nan')):.1f} & "
                    f"{(np.mean(c['temperature']) if c['temperature'] else float('nan')):.2f} \\\\\n")
        f.write(r"\bottomrule\end{tabular}\end{table}" "\n")

    # external validation
    ext = {}
    lines = []
    for ds in meta["externals"]:
        R = external_summary(data, a.reference, ds)
        if R is None:
            continue
        ext[ds] = {}
        for m in methods:
            E = external_summary(data, m, ds)
            if E is None:
                continue
            res = {k: E[k] for k in ("qwk", "qwk_ci", "qwk_runs_mean", "qwk_runs_sd", "accuracy",
                                     "auc_referable", "sev_nr", "sev_nr_ci", "sev_n", "pdr_nr", "pdr_nr_ci",
                                     "ece", "conf_0_4")}
            if m != a.reference and np.array_equal(E["images"], R["images"]):
                ca, cb = R["P"].argmax(1) == R["y"], E["P"].argmax(1) == E["y"]
                res["mcnemar_p"] = mcnemar_exact(ca, cb)
                Pj = np.concatenate([R["P"], E["P"]], 1)
                res["delta_qwk_ci"] = patient_bootstrap(R["y"], Pj, R["patients"],
                                                        lambda yy, PP: qwk_of(yy, PP[:, :5]) - qwk_of(yy, PP[:, 5:]))
            ext[ds][m] = res
            sev = f"{100 * res['sev_nr']:.1f} [{100 * res['sev_nr_ci'][0]:.1f}, {100 * res['sev_nr_ci'][1]:.1f}]"
            dci = res.get("delta_qwk_ci")
            name = tex_escape(m) if m != a.reference else r"\textbf{" + tex_escape(m) + "}"
            lines.append(f"{tex_escape(ds)} & {name} & {fmt(res['qwk'])} [{fmt(res['qwk_ci'][0])}, {fmt(res['qwk_ci'][1])}] & "
                         f"{fmt(res['auc_referable'])} & {sev} & {res['conf_0_4']} & "
                         f"{'--' if dci is None else f'[{dci[0]:+.3f}, {dci[1]:+.3f}]'} & "
                         f"{fmt_p(res.get('mcnemar_p', float('nan')))} \\\\")
    if lines:
        with open(out / "table_external.tex", "w") as f:
            f.write(r"\begin{table*}[t]\centering\small\caption{External validation: models trained on the internal "
                    r"data only, ensemble of the cross-validated models.}\label{tab:external}" "\n"
                    r"\resizebox{\textwidth}{!}{\begin{tabular}{@{}llcccccc@{}}\toprule Dataset & Method & QWK [95\% CI] & AUC (ref.) & "
                    r"Sev-NR \% [95\% CI] & 0$\leftrightarrow$4 & $\Delta$QWK (ours $-$ method) 95\% CI & McNemar $p$ \\ \midrule" "\n")
            f.write("\n".join(lines) + "\n" + r"\bottomrule\end{tabular}}\end{table*}" "\n")

    # literature (reported, different protocols)
    if a.literature:
        lit = list(csv.DictReader(open(a.literature)))
        need = ["method", "year", "citation_key", "dataset", "protocol", "metric", "value"]
        bad = [r for r in lit if any(not (r.get(k) or "").strip() for k in need)]
        if bad:
            raise SystemExit(f"literature rows with missing fields (every value needs a source and a protocol): {bad[:3]}")
        with open(out / "table_literature.tex", "w") as f:
            f.write(r"\begin{table*}[t]\centering\small\caption{Results reported in the literature. Values are copied "
                    r"from the cited papers; datasets, splits and metrics differ, so they are not directly comparable "
                    r"with Table~\ref{tab:sota}.}\label{tab:literature}" "\n"
                    r"\resizebox{\textwidth}{!}{\begin{tabular}{@{}llllll@{}}\toprule Method & Year & Dataset & Protocol & Metric & Value \\ \midrule" "\n")
            for r in lit:
                f.write(f"{tex_escape(r['method'])}~\\cite{{{r['citation_key']}}} & {r['year']} & {tex_escape(r['dataset'])} & "
                        f"{tex_escape(r['protocol'])} & {tex_escape(r['metric'])} & {tex_escape(r['value'])} \\\\\n")
            f.write(r"\bottomrule\end{tabular}}\end{table*}" "\n")

    # macros quoted in the text
    best_base = max(sota, key=lambda m: S[m]["qwk"][0]) if sota else None
    mac = {"OursQWK": fmt(ref["qwk"][0]), "OursQWKsd": fmt(ref["qwk"][1]), "OursQWKpooled": fmt(ref["qwk_pooled"]),
           "OursQWKlo": fmt(ref["qwk_ci"][0]), "OursQWKhi": fmt(ref["qwk_ci"][1]),
           "OursSevNR": f"{100 * ref['sev_nr']:.1f}", "OursSevNRhi": f"{100 * ref['sev_nr_ci'][1]:.1f}",
           "OursECE": fmt(ref["ece"][0]), "NSeeds": str(len(meta["args"]["seeds"])), "NFolds": str(meta["args"]["folds"])}
    if best_base:
        mac.update({"BestBaseline": tex_escape(best_base), "BestBaselineQWK": fmt(S[best_base]["qwk"][0]),
                    "DeltaBest": f"{S[best_base]['delta_qwk']:+.3f}", "PBest": fmt_p(padj[best_base])})
    for i, ds in enumerate(ext):
        if a.reference in ext[ds]:
            tag = "Ext" + "ABCDEFGH"[i]
            mac[tag + "Name"] = tex_escape(ds)
            mac[tag + "QWK"] = fmt(ext[ds][a.reference]["qwk"])
            mac[tag + "SevNR"] = f"{100 * ext[ds][a.reference]['sev_nr']:.1f}"
    with open(out / "results_macros.tex", "w") as f:
        f.write("% generated by analysis.py -- do not edit by hand\n")
        for k, v in mac.items():
            f.write(f"\\newcommand{{\\{k}}}{{{v}}}\n")

    # figures
    if not a.no_figures:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        order = sota + [a.reference] + abl
        fig, ax = plt.subplots(figsize=(7, 0.35 * len(order) + 1))
        for i, m in enumerate(order):
            lo, hi = S[m]["qwk_ci"]
            c = "#1F4E79" if m == a.reference else ("#888888" if m in abl else "#A7B4C2")
            q = S[m]["qwk_pooled"]
        ax.errorbar(q, i, xerr=[[max(q - lo, 0)], [max(hi - q, 0)]], fmt="o", color=c, capsize=3)
        ax.set_yticks(range(len(order)), order)
        ax.invert_yaxis()
        ax.set_xlabel("Pooled out-of-fold QWK (95% patient-bootstrap CI)")
        ax.grid(axis="x", alpha=0.3)
        fig.tight_layout()
        fig.savefig(out / "fig_qwk.pdf")
        plt.close(fig)
        cm = confusion_matrix(y, P.argmax(1), labels=range(5), normalize="true")
        fig, ax = plt.subplots(figsize=(4.2, 3.6))
        im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        for i in range(5):
            for j in range(5):
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cm[i, j] > 0.5 else "black")
        ax.set_xticks(range(5), GRADES, rotation=30)
        ax.set_yticks(range(5), GRADES)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Reference")
        fig.colorbar(im)
        fig.tight_layout()
        fig.savefig(out / "fig_confusion.pdf")
        plt.close(fig)
        conf, corr = P.max(1), P.argmax(1) == y
        edges = np.linspace(0, 1, 11)
        xs, ys_ = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            mk = (conf > lo) & (conf <= hi)
            if mk.any():
                xs.append(conf[mk].mean())
                ys_.append(corr[mk].mean())
        fig, ax = plt.subplots(figsize=(3.6, 3.6))
        ax.plot([0, 1], [0, 1], "--", color="gray")
        ax.plot(xs, ys_, "o-", color="#1F4E79")
        ax.set_xlabel("Confidence")
        ax.set_ylabel("Accuracy")
        ax.set_title(f"ECE = {ref['ece'][0]:.3f}")
        fig.tight_layout()
        fig.savefig(out / "fig_reliability.pdf")
        plt.close(fig)

    clean = {m: {k: v for k, v in s.items() if k != "pooled"} for m, s in S.items()}
    for m in clean:
        clean[m]["p_holm"] = padj.get(m)
    ext_clean = {ds: {m: {k: (list(v) if isinstance(v, tuple) else v) for k, v in r.items()} for m, r in d.items()}
                 for ds, d in ext.items()}
    json.dump({"internal": clean, "external": ext_clean, "reference": a.reference, "test_train_ratio": ratio},
              open(out / "results.json", "w"), indent=1, default=float)
    print(f"tables, macros and figures written to {out}")
    for m in sota + [a.reference] + abl:
        s = S[m]
        print(f"  {m:<36} QWK {s['qwk'][0]:.3f}±{s['qwk'][1]:.3f}  Sev-NR {100 * s['sev_nr']:.1f}%  "
              f"p_Holm={'ref' if m == a.reference else fmt_p(padj.get(m, float('nan')))}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile explain.py
"""Explanation figures for TAG-MTL-DR v2 (one panel per image).

For each image: (a) the fundus, (b) the vessel graph rooted at the optic disc, nodes
coloured by the attention they receive from the lesion tokens, (c) the lesion-token
attention map (how much each image region attends to the vessel structures), and
(d) the predicted grade probabilities. Intended for the qualitative figure of the paper
and for failure analysis (always show errors, not only successes).

Usage
-----
    python explain.py --checkpoint model.pt --cache cache/ --manifest manifest.csv \
        --images 0 15 42 --out figures/
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import torch

from tag_mtl_dr import (IMAGENET_MEAN, IMAGENET_STD, CachedFundus, TAGMTLDR, class_probs, collate,
                        gpu_preprocess)

GRADES = ["No DR", "Mild", "Moderate", "Severe", "PDR"]


@torch.no_grad()
def explain(model, x_uint8, graph, size, device):
    model.eval()
    x = gpu_preprocess(x_uint8[None], size, device)
    logits, _, attn = model(x, graph)
    P = class_probs(logits.float().cpu(), "coral")[0].numpy()
    if attn is None:
        return P, None, None
    a = attn[0].float().cpu().numpy()                    # (tokens, keys) averaged over heads
    n_nodes = graph.num_nodes
    node_att = a[:, :n_nodes].mean(0)                    # attention received by each vessel node
    tok = a[:, :n_nodes].sum(1)                          # share of each token's attention on vessel nodes
    h = int(np.sqrt(len(tok)))
    return P, node_att, tok.reshape(h, h)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--checkpoint", required=True, help="state_dict of a trained TAGMTLDR")
    ap.add_argument("--cache", required=True)
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--images", nargs="+", type=int, required=True, help="row indices of the manifest")
    ap.add_argument("--size", type=int, default=300)
    ap.add_argument("--backbone", default="efficientnet_b3")
    ap.add_argument("--out", default="figures")
    a = ap.parse_args()

    import csv
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    rows = list(csv.DictReader(open(a.manifest)))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TAGMTLDR(pretrained=False, backbone=a.backbone)
    model.load_state_dict(torch.load(a.checkpoint, map_location="cpu"))
    model.to(device)
    ds = CachedFundus(a.cache, a.images, [int(rows[i]["grade"]) for i in a.images])
    Path(a.out).mkdir(parents=True, exist_ok=True)
    for k, i in enumerate(a.images):
        x, y, _, g = ds[k]
        _, _, _, gb = collate([(x, y, torch.zeros(1), g)])
        P, node_att, tok = explain(model, x, gb, a.size, device)
        img = x.permute(1, 2, 0).numpy()
        S = img.shape[0]
        fig, ax = plt.subplots(1, 4, figsize=(15, 4))
        ax[0].imshow(img)
        ax[0].set_title(f"reference grade {y} ({GRADES[y]})")
        ax[1].imshow(img, alpha=0.35)
        pos = g.pos.numpy() * S
        for s_, d_ in g.edge_index.t().numpy():
            ax[1].plot(pos[[s_, d_], 1], pos[[s_, d_], 0], color="white", lw=0.8)
        c = node_att if node_att is not None else np.zeros(len(pos))
        sc = ax[1].scatter(pos[:, 1], pos[:, 0], c=c, s=18, cmap="magma")
        root = g.x[:, 6].numpy() > 0
        ax[1].scatter(pos[root, 1], pos[root, 0], marker="*", s=120, c="cyan", label="root (optic disc side)")
        ax[1].legend(loc="lower right", fontsize=7)
        fig.colorbar(sc, ax=ax[1], fraction=0.046)
        ax[1].set_title("vessel graph: attention received")
        if tok is not None:
            ax[2].imshow(img)
            ax[2].imshow(np.kron(tok, np.ones((S // tok.shape[0] + 1,) * 2))[:S, :S], alpha=0.5, cmap="jet")
        ax[2].set_title("lesion tokens: attention to vessels")
        ax[3].bar(range(5), P, color=["#999"] * 5)
        ax[3].bar([int(P.argmax())], [P.max()], color="#1F4E79")
        ax[3].set_xticks(range(5), GRADES, rotation=30)
        ax[3].set_ylim(0, 1)
        ax[3].set_title("predicted probabilities")
        for axx in ax[:3]:
            axx.axis("off")
        fig.tight_layout()
        fig.savefig(Path(a.out) / f"explain_{i}.pdf")
        plt.close(fig)
        print(f"image {i}: grade {y}, predicted {int(P.argmax())} (p = {P.max():.2f})")


if __name__ == "__main__":
    main()

In [ ]:
!python tag_mtl_dr.py --smoke

## 4. Data manifest

The benchmark reads a CSV with the columns `image,grade,patient,dataset,role[,mask]`:
* `role = internal` → cross-validated (patient-grouped folds);
* `role = external` → never used for training, only for evaluation;
* `patient` groups the two eyes of the same person (very important);
* `mask` (optional) → path of an annotated vessel mask; otherwise pseudo-labels are used.

The helpers below build it from the usual label files. Adapt the paths.

In [ ]:
import csv, glob, re
from pathlib import Path

def rows_from_label_csv(label_csv, image_dir, image_col, grade_col, dataset, role,
                        patient_fn=lambda stem: stem, ext=None):
    """One row per labelled image found on disk."""
    out, missing = [], 0
    with open(label_csv, newline="") as f:
        for r in csv.DictReader(f):
            stem = Path(r[image_col]).stem
            cands = [Path(image_dir) / r[image_col]] if ext is None else [Path(image_dir) / f"{stem}{ext}"]
            if ext is None and not cands[0].suffix:
                cands = [Path(image_dir) / f"{stem}{e}" for e in (".png", ".jpg", ".jpeg", ".tif")]
            path = next((c for c in cands if c.exists()), None)
            if path is None:
                missing += 1
                continue
            out.append({"image": str(path), "grade": int(float(r[grade_col])),
                        "patient": f"{dataset}:{patient_fn(stem)}", "dataset": dataset, "role": role})
    print(f"{dataset}: {len(out)} images, {len({o['patient'] for o in out})} patients, {missing} missing")
    return out

def write_manifest(rows, path):
    keys = ["image", "grade", "patient", "dataset", "role"] + (["mask"] if any("mask" in r for r in rows) else [])
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    print("manifest:", path, len(rows), "rows")

# patient identifiers
eyepacs_patient = lambda stem: re.match(r"^(\d+)_(left|right)$", stem).group(1)   # 10_left -> 10
one_image_per_patient = lambda stem: stem                                         # APTOS 2019

In [ ]:
MANIFEST = f"{WORK_DIR}/manifest.csv"

if DEMO:
    # tiny synthetic dataset (64 x 64 images) to check the pipeline -- numbers are meaningless
    import numpy as np
    from PIL import Image
    rng = np.random.default_rng(0)
    os.makedirs(f"{WORK_DIR}/img", exist_ok=True)
    def fake(g):
        a = rng.integers(0, 60, (64, 64, 3)).astype(np.uint8); a[..., 0] += np.uint8(30 * g)
        yy, xx = np.ogrid[:64, :64]; a[(abs(yy - 32) < 2) | (abs(xx - 20) < 2)] = 10
        return a
    rows = []
    for p in range(40):
        g = int(rng.integers(0, 5))
        for eye in "LR":
            f = f"{WORK_DIR}/img/A{p}{eye}.png"; Image.fromarray(fake(g)).save(f)
            rows.append({"image": f, "grade": g, "patient": f"A{p}", "dataset": "synthA", "role": "internal"})
    for p in range(20):
        g = int(rng.integers(0, 5)); f = f"{WORK_DIR}/img/B{p}.png"; Image.fromarray(fake(g)).save(f)
        rows.append({"image": f, "grade": g, "patient": f"B{p}", "dataset": "synthB", "role": "external"})
    write_manifest(rows, MANIFEST)
else:
    rows = []
    # --- EXAMPLES: uncomment and adapt to your folders ---
    # APTOS 2019 (one image per patient), internal:
    # rows += rows_from_label_csv(f"{DATA_DIR}/aptos2019/train.csv", f"{DATA_DIR}/aptos2019/train_images",
    #                             "id_code", "diagnosis", "APTOS2019", "internal", one_image_per_patient, ".png")
    # Kaggle EyePACS (two eyes per patient), external:
    # rows += rows_from_label_csv(f"{DATA_DIR}/eyepacs/trainLabels.csv", f"{DATA_DIR}/eyepacs/train",
    #                             "image", "level", "EyePACS", "external", eyepacs_patient, ".jpeg")
    # Messidor-2: provide a CSV image,grade,patient (patient = exam / person id), external:
    # rows += rows_from_label_csv(f"{DATA_DIR}/messidor2/labels.csv", f"{DATA_DIR}/messidor2/images",
    #                             "image", "grade", "Messidor2", "external", lambda s: s)
    assert rows, "add your datasets above"
    write_manifest(rows, MANIFEST)

## 5. Benchmark
Every method uses the same patient-grouped folds, seeds, budget and pre-processing. Predictions are appended to `predictions.csv` after each run (safe if Colab disconnects: re-run with the remaining methods into another folder).

In [ ]:
RUNS = f"{WORK_DIR}/runs/main"
methods = " ".join(f'"{m}"' for m in METHODS)
seeds = " ".join(map(str, SEEDS))
extra = ("--no_pretrained --backbone_override resnet18 --cache_size 64 --epochs 2 --patience 2" if DEMO
         else f"--epochs {EPOCHS}") + (" --save_models" if SAVE_MODELS else "")
cache = CACHE_DIR + ("_demo" if DEMO else "")
folds = 2 if DEMO else FOLDS
!python benchmark.py --manifest "{MANIFEST}" --cache "{cache}" --out "{RUNS}" --methods {methods} --seeds {seeds} --folds {folds} --batch {BATCH} --workers {WORKERS} {extra}

## 6. Statistics, tables and figures
Optional: upload `literature_reported.csv` (columns `method,year,citation_key,dataset,protocol,metric,value`) with values copied from papers **with their source**; it becomes a separate table.

In [ ]:
LIT = f"{WORK_DIR}/literature_reported.csv"
lit_arg = f'--literature "{LIT}"' if os.path.exists(LIT) else ""
!python analysis.py --runs "{RUNS}" --reference "TAG-MTL-DR (ours)" {lit_arg}

In [ ]:
import json, pandas as pd
from IPython.display import display, IFrame
res = json.load(open(f"{RUNS}/paper/results.json"))
tab = pd.DataFrame({m: {"QWK mean": s["qwk"][0], "QWK sd": s["qwk"][1], "pooled QWK": s["qwk_pooled"],
                        "CI low": s["qwk_ci"][0], "CI high": s["qwk_ci"][1], "Sev-NR %": 100 * s["sev_nr"],
                        "ECE": s["ece"][0], "p Holm": s.get("p_holm")} for m, s in res["internal"].items()}).T
display(tab.round(3))
for ds, d in res["external"].items():
    print("External:", ds)
    display(pd.DataFrame({m: {"QWK": r["qwk"], "Sev-NR %": 100 * r["sev_nr"], "McNemar p": r.get("mcnemar_p")}
                          for m, r in d.items()}).T.round(3))

In [ ]:
from IPython.display import Image as Img
!pip -q install pdf2image > /dev/null; apt-get -qq install -y poppler-utils > /dev/null
from pdf2image import convert_from_path
for f in ["fig_qwk.pdf", "fig_confusion.pdf", "fig_reliability.pdf"]:
    convert_from_path(f"{RUNS}/paper/{f}", dpi=110)[0].save(f"/content/{f}.png"); display(Img(f"/content/{f}.png"))
print(open(f"{RUNS}/paper/table_sota.tex").read())

## 7. Explanations (successes *and* failures)
Uses a saved checkpoint of the proposed model.

In [ ]:
import glob
ck = sorted(glob.glob(f"{RUNS}/checkpoints/TAG-MTL-DR_(ours)_*.pt"))
if ck:
    ids = "0 1 2 3" if DEMO else "0 1 2 3 4 5"      # choose manifest rows, e.g. misclassified severe images
    size = 64 if DEMO else 300
    bb = "--backbone resnet18" if DEMO else ""
    !python explain.py --checkpoint "{ck[0]}" --cache "{cache}" --manifest "{MANIFEST}" --images {ids} --size {size} {bb} --out "{RUNS}/paper/explanations"
else:
    print("no checkpoint: set SAVE_MODELS = True and re-run the benchmark")

## 8. Download everything
The ZIP contains the predictions, `results.json`, the LaTeX tables, the macros and the figures, ready for the paper skeleton (`paper/main.tex` of the repository).

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/content/tag_mtl_dr_results", "zip", RUNS)
from google.colab import files
files.download(zip_path)